In [ ]:
import numpy
import pandas
from src.data_objects import AllData
from src.functions import *
from src.readers import ReadDesign
import os

retrain = True
runchain = True
logTrain = True
scikit = False

ThisData = AllData
ThisData["name"] = "pp-only"

model_par = "input/ppDesign.txt"
outdir = "/data/rjfgroup/rjf01/cameron.parker/tunes/" + ThisData["name"] + "/"
ThisData["Design"] = ReadDesign("/data/rjfgroup/rjf01/cameron.parker/runs/no-part-prop/LHC2760/QVir_Analysis/parameters.txt")

try:
    os.mkdir(outdir)
except:
    print("Output dir exists, will be overwritten")

Trimming points and obs ranges

In [ ]:
#doesnt converge
#del ThisData["Observables"]["PrPr200"]["kaon-pT-soft"]
del ThisData["Observables"]["PrPr200"]["proton-pT-soft"]
del ThisData["Observables"]["PrPr200"]["pion-pT-hard"]

#del ThisData["Observables"]["PrPr2760"]["charged-pT-soft"]
#del ThisData["Observables"]["PrPr2760"]["charged-pT-hard"]
#del ThisData["Observables"]["PrPr2760"]["pion-pT-hard"]
#del ThisData["Observables"]["PrPr2760"]["kaon-pT-soft"]
#del ThisData["Observables"]["PrPr2760"]["kaon-pT-hard"]
del ThisData["Observables"]["PrPr2760"]["proton-pT-soft"]
del ThisData["Observables"]["PrPr2760"]["proton-pT-hard"]

del ThisData["Observables"]["PrPr13000"]
del ThisData["Observables"]["EpEm91"]

#del ThisData["Observables"]["PrPr200"]
#del ThisData["Observables"]["PrPr2760"]
#del ThisData["Observables"]["PrPr13000"]
#del ThisData["Observables"]["EpEm91"]

ThisData["Observables"]["PrPr200"]["pion-pT-soft"]["cuts"] = []
ThisData["Observables"]["PrPr200"]["kaon-pT-soft"]["cuts"] = []

trimRanges(ThisData)

In [ ]:
badpoints = []
trimPoints(badpoints,ThisData)

Making Data pkl for selected observables

In [ ]:
buildDataPkl(ThisData, logTrain)
print(ThisData["datapkl"])

Getting emulators

In [ ]:
from src.emulator_BAND import EmulatorBAND

setEmuPaths(ThisData)

if retrain:
    buildObsPkls(ThisData)
    trainEmulators(model_par, ThisData, logTrain, scikit=scikit)
else:
    readEmulators(ThisData)

Running Chain

In [ ]:
from src.mcmc import Chain
import os

mcmcpath = "mcmc/" + ThisData["name"] + "-chain.pkl"
mymcmc = Chain(mcmc_path=mcmcpath, expdata_path=ThisData["datapkl"], model_parafile=model_par)
mymcmc.loadEmulator(getEmuPathList(ThisData))

In [ ]:
print(mymcmc.max)

In [ ]:
os.environ["OMP_NUM_THREADS"] = "20"
# may have to: export RDMAV_FORK_SAFE=1 before running the code

n_effective=16000
n_active=8000
n_prior=16000
sample="tpcn"
n_max_steps=500
random_state=43

n_total = 100000
n_evidence = 0

pool = 20

if runchain:
    sampler = mymcmc.run_pocoMC(n_effective=n_effective, n_active=n_active,
                            n_prior=n_prior, sample=sample,
                            n_max_steps=n_max_steps, random_state=random_state,
                            n_total=n_total, n_evidence=n_evidence, pool=pool)

Corner Plot

In [ ]:
import pickle
import corner
import matplotlib.pyplot as plt
import numpy as np
        
with open(mcmcpath, 'rb') as pf:
        data = pickle.load(pf)

labels = mymcmc.label

fig = corner.corner(data['chain'], weights=data['weights'], labels=labels, color="C0")
plt.show()

chain_params(data,ThisData["Design"]["Parameter"],outdir)

In [ ]:
print(data["chain"])

In [ ]:
TransformedSamples = np.copy(data['chain'])
TransformedSamples[:,0] = data['chain'][:,0]
TransformedSamples[:,1] = data['chain'][:,1]
TransformedSamples[:,2] = data['chain'][:,2]
TransformedSamples[:,3] = (2*data['chain'][:,6]+0.05) + (data['chain'][:,2]-(2*data['chain'][:,6]+0.05))*data['chain'][:,3]
TransformedSamples[:,4] = data['chain'][:,4]
TransformedSamples[:,5] = data['chain'][:,5]
TransformedSamples[:,6] = data['chain'][:,6]

labels[3] = "QS"
fig = corner.corner(TransformedSamples, weights=data['weights'], labels=labels, color="C0")
plt.show()
fig.savefig(outdir+'Corner.pdf', dpi = 192)

Priors

In [ ]:
makeplot(ThisData, "Priors", outdir, logTrain=logTrain)

Posteriors

In [ ]:
makeplot(ThisData, "Posteriors", outdir, samples=data["chain"], logTrain=logTrain, scikit=scikit)

Validation

In [ ]:
from src.data_objects import valData

valData["Design"] = ReadDesign("/data/rjfgroup/rjf01/cameron.parker/runs/no-part-prop-val/LHC2760/QVir_Analysis/parameters.txt")
updateCuts(ThisData,valData)
trimRanges(valData)
validationPlots(valData, ThisData, outdir, logTrain=logTrain, scikit=scikit)

Honesty Plots

In [ ]:
ErrorHonestyPlots(valData,ThisData,outdir,logTrain=logTrain,scikit=scikit)

Closure Tests

In [ ]:
closureTest(ThisData, valData, outdir, model_par, runchain=True, logTrain=logTrain)